In [ ]:
from fastcore.all import *
from iomeval.readers import find_eval, load_evals, eval_url, get_report_urls
from mistocr.core import read_pgs

import pandas as pd
from iomeval.pipeline import run_pipeline, batch_run

pd.set_option('display.max_colwidth', None)

In [ ]:
models = [
    'claude-sonnet-4-5',
    'claude-sonnet-4-6', 
    'claude-opus-4-5-20251101'
    ]

In [ ]:
#DATA_PATH = Path('../../data')

In [ ]:
DATA_PATH = Path().home() / 'iomeval/data'

In [ ]:
#EVALS_PATH = Path('../files/test/evaluations.json')

In [ ]:
EVALS_PATH = Path().home() / 'iomeval/nbs/files/test/evaluations-research.json'

In [ ]:
evals = load_evals(EVALS_PATH)

In [ ]:
!ls ../../../tagapp/backups/20260311_115159/data 

06fd0949dc874f388686461c19d2669c.json  760568a15e144bcfb88dbbd50247dc51.json
0d4309b702e00faac631be8d609934ce.json  891448b7a4bf12763ea0579dd67e289d.json
1d7e6d2d445145d1d350b48b72b36681.json  8dd2261100ff5d714e73b411eed043d5.json
22cac1c000836253adc445993e101560.json  9992310969aa2f428bc8aba29f865cf3.json
316c47c9dc19c820f3217434d4b93b28.json  a493c447cff3e03ca94024335d5c5c7a.json
4341695461234eee3deb51ac68871109.json  dba9eb1220368d28682d8531108e767d.json
49d2fba781b6a7c0d94577479636ee6f.json  e21305a89aeb76fafa8f4270392d463a.json
4c04ec0d3e651f7616190c6fc2dcfc18.json  e8d9427f4a9094f68e17307faa567d8b.json
56e6f2bc9f55a0c6d07341ddde1283db.json  f409b3f9a0671ec69a8591c425a45c1d.json
6c3c2cf3fa479112967612b0baddab72.json  legacy
75bad4665ed458cce78740e3d86f72d0.json  tagapp.db


In [ ]:
#LATEST_TAG_PATH = Path('../../../tagapp/backups/20260311_115159/data')

In [ ]:
LATEST_TAG_PATH = Path.home() / 'tagapp/backups/20260311_115159/data'

In [ ]:
def get_deployed_ids(backup_path):
    "List of report IDs currently deployed (from tagapp backup)"
    return Path(backup_path).ls(file_exts='.json').map(lambda p: p.stem)

In [ ]:
current_res = get_deployed_ids(LATEST_TAG_PATH)
current_res

['8dd2261100ff5d714e73b411eed043d5', '1d7e6d2d445145d1d350b48b72b36681', '760568a15e144bcfb88dbbd50247dc51', '22cac1c000836253adc445993e101560', '4341695461234eee3deb51ac68871109', '9992310969aa2f428bc8aba29f865cf3', '316c47c9dc19c820f3217434d4b93b28', '0d4309b702e00faac631be8d609934ce', 'e8d9427f4a9094f68e17307faa567d8b', '75bad4665ed458cce78740e3d86f72d0', 'f409b3f9a0671ec69a8591c425a45c1d', 'dba9eb1220368d28682d8531108e767d', '56e6f2bc9f55a0c6d07341ddde1283db', '49d2fba781b6a7c0d94577479636ee6f', 'a493c447cff3e03ca94024335d5c5c7a', '06fd0949dc874f388686461c19d2669c', '6c3c2cf3fa479112967612b0baddab72', '891448b7a4bf12763ea0579dd67e289d', '4c04ec0d3e651f7616190c6fc2dcfc18', 'e21305a89aeb76fafa8f4270392d463a']

In [ ]:
def load_result(eval_id, base_path):
    "Load a result JSON by eval_id from base_path"
    return (Path(base_path)/f'{eval_id}.json').read_json()


In [ ]:
load_result('8dd2261100ff5d714e73b411eed043d5', DATA_PATH / 'results').keys()

dict_keys(['id', 'report_url', 'meta', 'docs', 'curation_status', 'selected_headings', 'mappings', 'timestamp'])

In [ ]:
def is_remapped(eval_id, base_path, backup_path):
    "Check if report mappings differ between current results and deployed backup"
    current = load_result(eval_id, base_path)
    deployed = load_result(eval_id, backup_path)
    return current.get('mappings') != deployed.get('mappings')

In [ ]:
is_remapped('8dd2261100ff5d714e73b411eed043d5', DATA_PATH/'results', LATEST_TAG_PATH)

True

In [ ]:
test_id = current_res[0]
test_id

'8dd2261100ff5d714e73b411eed043d5'

##  Tagging

In [ ]:
#result = await batch_run(evals, ids=[test_id], base_path=str(DATA_PATH), force={'enbs','ccps','gcms','outs'})

iomeval.pipeline - INFO - 8dd22611... completed


In [ ]:
result = await batch_run(evals, ids=[test_id], base_path=str(DATA_PATH), 
                         force={'enbs','ccps','gcms','outs'}, verbose=True, model=models[1])

iomeval.pipeline - INFO - Mapping enablers...


iomeval.pipeline - INFO - Mapping CCPs...


iomeval.pipeline - INFO - Mapping GCM objectives...


iomeval.pipeline - INFO - Mapping outputs...


iomeval.pipeline - INFO - Pipeline complete!


iomeval.pipeline - INFO - 8dd22611... completed


In [ ]:
[d for d in current_res if not is_remapped(d, DATA_PATH/'results', LATEST_TAG_PATH)]


['1d7e6d2d445145d1d350b48b72b36681',
 '9992310969aa2f428bc8aba29f865cf3',
 '316c47c9dc19c820f3217434d4b93b28',
 '0d4309b702e00faac631be8d609934ce',
 'dba9eb1220368d28682d8531108e767d',
 '56e6f2bc9f55a0c6d07341ddde1283db',
 'e21305a89aeb76fafa8f4270392d463a']

In [ ]:
def load_batch_run(base_path, n=1, timestamp=None):
    "Load a batch run log by timestamp, or the n latest if None (1 returns single item, n>1 returns list)"
    if timestamp is not None:
        p = base_path / 'batch_runs' / f'{timestamp}.json'
        return p.stem, p.read_json()
    runs = sorted((base_path / 'batch_runs').ls())
    if n == 1: return runs[-1].stem, runs[-1].read_json()
    return [(p.stem, p.read_json()) for p in runs[-n:]]

In [ ]:
load_batch_run(DATA_PATH, n=2)

[('2026-04-21_07-23',
  {'completed': [],
   'awaiting_curation': ['19b8bb65e753a0178015c5330e113bd0',
    'c9cacbb73cbcba9f3a8758b8fc3f66e0',
    'a61bcf10a02495e017eadaf45d175ab9',
    'b1f220cc12464c9599ebb536bd5dbc3f'],
   'failed': [],
   'skipped': []}),
 ('2026-04-22_08-44',
  {'completed': [],
   'awaiting_curation': ['11111111111111111111111111111111',
    '21111111111111111111111111111111',
    '31111111111111111111111111111111',
    '41111111111111111111111111111111',
    '51111111111111111111111111111111'],
   'failed': [],
   'skipped': []})]

In [ ]:
current_res[1:]

['1d7e6d2d445145d1d350b48b72b36681', '760568a15e144bcfb88dbbd50247dc51', '22cac1c000836253adc445993e101560', '4341695461234eee3deb51ac68871109', '9992310969aa2f428bc8aba29f865cf3', '316c47c9dc19c820f3217434d4b93b28', '0d4309b702e00faac631be8d609934ce', 'e8d9427f4a9094f68e17307faa567d8b', '75bad4665ed458cce78740e3d86f72d0', 'f409b3f9a0671ec69a8591c425a45c1d', 'dba9eb1220368d28682d8531108e767d', '56e6f2bc9f55a0c6d07341ddde1283db', '49d2fba781b6a7c0d94577479636ee6f', 'a493c447cff3e03ca94024335d5c5c7a', '06fd0949dc874f388686461c19d2669c', '6c3c2cf3fa479112967612b0baddab72', '891448b7a4bf12763ea0579dd67e289d', '4c04ec0d3e651f7616190c6fc2dcfc18', 'e21305a89aeb76fafa8f4270392d463a']

In [ ]:
ids_to_tag = L(evals).map(lambda d: d.id)[1:]
ids_to_tag

['21111111111111111111111111111111', '31111111111111111111111111111111', '41111111111111111111111111111111', '51111111111111111111111111111111']

In [ ]:
result = await batch_run(evals, ids=ids_to_tag, base_path=str(DATA_PATH), 
                         force={'enbs','ccps','gcms','outs'}, verbose=False, model='claude-sonnet-4-6')


iomeval.pipeline - INFO - 21111111... completed


iomeval.pipeline - INFO - 31111111... completed


iomeval.pipeline - INFO - 41111111... completed


iomeval.pipeline - INFO - 51111111... completed


In [ ]:
current_res = ids_to_tag
current_res

['21111111111111111111111111111111', '31111111111111111111111111111111', '41111111111111111111111111111111', '51111111111111111111111111111111']

In [ ]:
DATA_PATH /'results'/f'{current_res[0]}.json'

Path('/app/data/iomeval/data/results/21111111111111111111111111111111.json')

In [ ]:
!ls ../../../tagapp/tagapp/data

06fd0949dc874f388686461c19d2669c.json  891448b7a4bf12763ea0579dd67e289d.json
0d4309b702e00faac631be8d609934ce.json  8dd2261100ff5d714e73b411eed043d5.json
1d7e6d2d445145d1d350b48b72b36681.json  9992310969aa2f428bc8aba29f865cf3.json
22cac1c000836253adc445993e101560.json  a493c447cff3e03ca94024335d5c5c7a.json
316c47c9dc19c820f3217434d4b93b28.json  dba9eb1220368d28682d8531108e767d.json
4341695461234eee3deb51ac68871109.json  e21305a89aeb76fafa8f4270392d463a.json
49d2fba781b6a7c0d94577479636ee6f.json  e8d9427f4a9094f68e17307faa567d8b.json
4c04ec0d3e651f7616190c6fc2dcfc18.json  f409b3f9a0671ec69a8591c425a45c1d.json
56e6f2bc9f55a0c6d07341ddde1283db.json  legacy
6c3c2cf3fa479112967612b0baddab72.json  tagapp.db
75bad4665ed458cce78740e3d86f72d0.json  tagapp.db-shm
760568a15e144bcfb88dbbd50247dc51.json  tagapp.db-wal


In [ ]:
dest_dir = Path('../../../tagapp/tagapp/data')

In [ ]:
import shutil
def cp_results(ids, src, dest):
    for id in ids: shutil.copy(src/f'{id}.json', dest/f'{id}.json')

In [ ]:
TAGAPP_DATA = Path('../../../tagapp/tagapp/data')
cp_results(current_res, DATA_PATH/'results', TAGAPP_DATA)

In [ ]:
dest_dir = Path('../../data/results_new')
dest_dir.ls()

[Path('../../data/results_new/31111111111111111111111111111111.json'), Path('../../data/results_new/21111111111111111111111111111111.json'), Path('../../data/results_new/41111111111111111111111111111111.json'), Path('../../data/results_new/51111111111111111111111111111111.json')]

In [ ]:
current_res

['21111111111111111111111111111111', '31111111111111111111111111111111', '41111111111111111111111111111111', '51111111111111111111111111111111']

In [ ]:
cp_results(current_res, DATA_PATH/'results', dest_dir)